# MM-CAD B-Rep: normalization stats + reference example

Run once on Colab with Drive mounted. Produces in `OUT_DIR`:
* `brep_norm_stats_v4.npz` - `ff_mean`, `ff_std` (16,) exactly as used to train `baseline_trimodal_v4`
* `reference/` - for a few parts: raw tensors, model-ready tensors, 768-d embedding, (+ STEP and re-extraction check if the STEP is found)

It also **verifies** the fixed `inference.py` B-Rep forward against the stored v4 anchors (`sketch_anchors/brep_anchors.pt`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/MMCAD'
BREP_H5 = f'{DRIVE}/brep_features.h5'
TOPO_H5 = f'{DRIVE}/brep_topology.h5'
CKPT    = f'{DRIVE}/baseline_trimodal_v4/latest.pth'      # or checkpoints/baseline_trimodal_v4.pth from HF
ANCHORS = f'{DRIVE}/sketch_anchors/brep_anchors.pt'      # optional, for verification
STEP_DIR = None   # set to a folder of ABC STEP files named <uid>*.step to also check re-extraction
OUT_DIR = f'{DRIVE}/release_brep_ref'
REF_UIDS = None   # None -> 3 parts from the h5 with 10-60 faces
import os; os.makedirs(f'{OUT_DIR}/reference', exist_ok=True)
for p in [BREP_H5, TOPO_H5, CKPT, ANCHORS]: print(os.path.exists(p), p)

In [ ]:
!git clone -q https://github.com/exanos/MMCAD.git /content/MMCAD 2>/dev/null || (cd /content/MMCAD && git pull -q)
import sys; sys.path.insert(0, '/content/MMCAD'); sys.path.insert(0, '/content/MMCAD/scripts')

In [ ]:
# --- 1. Normalization stats: identical to TriModalDataset in mmcad_baseline_trimodal_v4 ---
# all samples in the h5, every face row that is not all-zero, population std, std<1e-6 -> 1
import h5py, numpy as np
with h5py.File(BREP_H5, 'r') as f:
    ff = f['face_features']
    n_feat = ff.shape[-1]
    s1 = np.zeros(n_feat); s2 = np.zeros(n_feat); cnt = 0
    for i in range(0, ff.shape[0], 5000):
        c = ff[i:i+5000].reshape(-1, n_feat)
        v = c[np.any(c != 0, axis=1)].astype(np.float64)
        s1 += v.sum(0); s2 += (v**2).sum(0); cnt += len(v)
    N = ff.shape[0]
mean = s1 / cnt
ff_mean = mean.astype(np.float32)
ff_std = np.sqrt(np.maximum(s2/cnt - mean**2, 0)).astype(np.float32)
ff_std[ff_std < 1e-6] = 1.0
np.savez(f'{OUT_DIR}/brep_norm_stats_v4.npz', ff_mean=ff_mean, ff_std=ff_std,
         n_samples=N, n_face_rows=cnt)
np.set_printoptions(precision=6, suppress=True)
print('samples', N, 'face rows', cnt); print('ff_mean', ff_mean); print('ff_std ', ff_std)

In [ ]:
# --- 2. BRepFormer only (no text model / HF login needed), strict load ---
import torch, inference as I
sd = torch.load(CKPT, map_location='cpu', weights_only=False)
sd = sd.get('model_state_dict', sd)
bsd = {k[len('brep_encoder.'):]: v for k, v in sd.items() if k.startswith('brep_encoder.')}
enc = I.BRepFormerEncoder(16, 768, 512, 8, 6, 192)
enc.load_state_dict(bsd, strict=True); enc.eval().to(I.DEVICE)
class _M: brep_encoder = enc
model = _M(); del sd
print('loaded', len(bsd), 'brep tensors strictly')

In [ ]:
# --- 3. Reference parts: raw -> prepared -> embedding ---
with h5py.File(BREP_H5, 'r') as f, h5py.File(TOPO_H5, 'r') as t:
    uids = [u.decode() if isinstance(u, bytes) else str(u) for u in f['uids'][:]]
    tuids = {(u.decode() if isinstance(u, bytes) else str(u)).strip(): i for i, u in enumerate(t['uids'][:])}
    nfa = f['num_faces'][:]
    if REF_UIDS is None:
        cand = [i for i in np.where((nfa >= 10) & (nfa <= 60))[0] if uids[i].strip() in tuids]
        idxs = cand[:3]
    else:
        idxs = [uids.index(u) for u in REF_UIDS]
    refs = {}
    for i in idxs:
        u = uids[i].strip()
        refs[u] = dict(face_features=f['face_features'][i], face_centroids=f['face_centroids'][i],
                       edge_to_faces=f['edge_to_faces'][i], face_mask=f['face_masks'][i],
                       face_normals=t['face_normals'][tuids[u]])
anchors = torch.load(ANCHORS, map_location='cpu') if os.path.exists(ANCHORS) else {}
for u, raw in refs.items():
    x = I.prepare_brep(raw, ff_mean, ff_std)
    z = I.embed_brep(model, x)                       # unit-norm (1,768)
    with torch.no_grad(): zr = model.brep_encoder(*[x[k].to(I.DEVICE) for k in ['face_features','face_centroids','edge_to_faces','face_mask','face_normals']])
    np.savez(f'{OUT_DIR}/reference/{u}.npz',
             **{f'raw_{k}': v for k, v in raw.items()},
             **{f'in_{k}': v[0].numpy() for k, v in x.items()},
             embedding_raw=zr[0].detach().cpu().numpy(), embedding=z[0].cpu().numpy())
    msg = ''
    if u in anchors:
        a = torch.as_tensor(anchors[u]).float().flatten()
        msg = f'  cos vs stored v4 anchor = {torch.nn.functional.cosine_similarity(a, zr[0].detach().cpu().float(), dim=0):.6f}'
    print(u, int(raw['face_mask'].sum()), 'faces', msg)

In [ ]:
# --- 4. (optional) STEP -> tensors re-extraction check ---
import glob, step_to_brep  # needs pythonocc-core: !pip install -q pythonocc-core  (or condacolab)
if STEP_DIR:
    for u, raw in refs.items():
        hits = glob.glob(f'{STEP_DIR}/**/{u}*.st*p', recursive=True)
        if not hits: print(u, 'no STEP found'); continue
        r = step_to_brep.extract(hits[0])
        for k in ['face_features','face_centroids','edge_to_faces','face_mask','face_normals']:
            print(u, k, 'max|diff| =', float(np.abs(r[k].astype(np.float64) - raw[k]).max()))
        import shutil; shutil.copy(hits[0], f'{OUT_DIR}/reference/{u}.step')